In [1]:
import numpy as np
import time
from warehouse_gridworld_domain_random import WarehouseGridWorld, ACTIONS, setup_pygame, draw_grid

pygame 2.6.1 (SDL 2.28.4, Python 3.11.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
def exploration_function(Q_sa, N_sa, k):
    return Q_sa + k / ((N_sa + 1) ** 0.5)

In [3]:
def select_action_with_exploration(Q, N, state_idx, valid_actions, k):
    best_action = None
    best_value = -float("inf")

    for a_idx, action in enumerate(ACTIONS):
        if action not in valid_actions:
            continue

        value = exploration_function(Q[state_idx, a_idx],
                                     N[state_idx, a_idx],
                                     k)

        if value > best_value:
            best_value = value
            best_action = a_idx

    return best_action

In [4]:
def train_q_learning_with_exploration(env, episodes=500, alpha=0.1, gamma=0.95, k=1.0):
    state_size = env.get_state_space_size()
    action_size = env.get_action_space_size()

    Q = np.zeros((state_size, action_size))
    N = np.zeros((state_size, action_size))  # visit counts

    rewards_per_episode = []

    for ep in range(episodes):
        state = env.reset()
        state_idx = env.state_to_index(state)

        total_reward = 0

        while not env.is_terminal():
            valid_actions = env.valid_actions()

            # select action using exploration function
            action_idx = select_action_with_exploration(Q, N, state_idx, valid_actions, k)
            action = ACTIONS[action_idx]

            result = env.step(action)
            next_state_idx = env.state_to_index(result.state)

            # increment visit count
            N[state_idx, action_idx] += 1

            # Q-learning update
            best_next = np.max(Q[next_state_idx])
            target = result.reward + (0 if result.done else gamma * best_next)

            Q[state_idx, action_idx] += alpha * (target - Q[state_idx, action_idx])

            state_idx = next_state_idx
            total_reward += result.reward

        rewards_per_episode.append(total_reward)

    return Q, N, rewards_per_episode

In [5]:
"""def train_q_learning_with_exploration(env, episodes=500, alpha=0.1, gamma=0.95, k=1.0):
    state_size = env.get_state_space_size()
    action_size = env.get_action_space_size()

    Q = np.zeros((state_size, action_size))
    N = np.zeros((state_size, action_size))  # visit counts

    rewards_per_episode = []

    for ep in range(episodes):
        state = env.reset()
        state_idx = env.state_to_index(state)

        total_reward = 0

        while not env.is_terminal():
            valid_actions = env.valid_actions()

            action_idx = select_action_with_exploration(Q, N, state_idx, valid_actions, k)
            action = ACTIONS[action_idx]

            result = env.step(action)
            next_state_idx = env.state_to_index(result.state)

            # display what's going on
            env.display() 
            print(f"Action: {action}, Reward: {result.reward}, Info: {result.info}")
            time.sleep(0.1)
    
            # increment visit count
            N[state_idx, action_idx] += 1

            # Q-learning update
            best_next = np.max(Q[next_state_idx])
            target = result.reward + (0 if result.done else gamma * best_next)

            Q[state_idx, action_idx] += alpha * (target - Q[state_idx, action_idx])

            state_idx = next_state_idx
            total_reward += result.reward

        rewards_per_episode.append(total_reward)

    return Q, N, rewards_per_episode
"""

'def train_q_learning_with_exploration(env, episodes=500, alpha=0.1, gamma=0.95, k=1.0):\n    state_size = env.get_state_space_size()\n    action_size = env.get_action_space_size()\n\n    Q = np.zeros((state_size, action_size))\n    N = np.zeros((state_size, action_size))  # visit counts\n\n    rewards_per_episode = []\n\n    for ep in range(episodes):\n        state = env.reset()\n        state_idx = env.state_to_index(state)\n\n        total_reward = 0\n\n        while not env.is_terminal():\n            valid_actions = env.valid_actions()\n\n            action_idx = select_action_with_exploration(Q, N, state_idx, valid_actions, k)\n            action = ACTIONS[action_idx]\n\n            result = env.step(action)\n            next_state_idx = env.state_to_index(result.state)\n\n            # display what\'s going on\n            env.display() \n            print(f"Action: {action}, Reward: {result.reward}, Info: {result.info}")\n            time.sleep(0.1)\n    \n            # increm

In [6]:
def run_learned_policy(env, Q):
    import pygame

    screen, clock, font, small_font = setup_pygame()

    state = env.reset()
    state_idx = env.state_to_index(state)

    running = True

    while not env.is_terminal() and running:

        # 👇 event loop (always first)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        # 👇 SELECT ACTION (THIS is where your code goes)
        valid = env.valid_actions()

        valid_indices = [i for i, a in enumerate(ACTIONS) if a in valid]
        action_idx = max(valid_indices, key=lambda i: Q[state_idx, i])

        action = ACTIONS[action_idx]

        # step environment
        result = env.step(action)
        state_idx = env.state_to_index(result.state)

        # 👇 render AFTER step
        draw_grid(env, screen, font, small_font)
        pygame.display.flip()
        clock.tick(5)

    pygame.quit()

In [7]:
env = WarehouseGridWorld()

Q1, N1, rewards1 = train_q_learning_with_exploration(env, k=1.0)

In [8]:
run_learned_policy(env, Q1)